# Breast histopathology classification with ResNet-18

**Zirui Chen · Deep learning coursework, HW6.2 · Portfolio edition**

Binary classification of H&E breast tissue images: normal (0) versus invasive carcinoma (1). This notebook presents the original transfer-learning experiment, separate course-validation evaluation, unlabeled-image prediction and Grad-CAM examples.

**Reading guide:** start with the recorded results below, then read Sections 1–7 for the implementation. Code has been reorganized, not rerun. Raw images, trained weights and original package versions are unavailable in this repository. See [data provenance](../data/README.md) and [method notes](../docs/method_notes.md).

## Recorded results — original run

The separately supplied course-validation set contains 82 images. The saved confusion matrix is `[[32, 3], [1, 46]]` (rows: true normal/IC; columns: predicted normal/IC). Accuracy was **78/82 (95.12%)**, IC sensitivity **97.87%**, normal specificity **91.43%**, and IC precision **93.88%**. These are coursework results, not clinical validation.

![Recorded validation confusion matrix](../figures/validation_confusion_matrix_original.png)

The 149 unlabeled images produced 59 normal and 90 IC predictions; no test accuracy can be calculated without labels.

## 1. Environment and paths

Install `requirements.txt` and open this notebook from the repository root or `notebooks/`. Populate the folders described in `data/README.md` before running. Executing all cells will train a model; it is not needed to read the archived results. Future outputs are kept apart from the recorded results.

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'data' / 'README.md').exists():
    raise RuntimeError('Open this notebook from the repository root or notebooks directory.')
DATA_DIR = ROOT / 'data' / 'raw'
CHECKPOINT_PATH = ROOT / 'models' / 'best_model.pth'
RESULT_DIR = ROOT / 'results' / 'generated'
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, Dataset
import os
from PIL import Image
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


## 2. Dataset and preprocessing

Labels are read from `0_N` and `1_IC`. Images are converted to RGB, resized to 224 × 224, and ImageNet-normalized. The original augmentation definition is retained below, but the shared-dataset assignment in Section 5 overwrites it for both subsets. Do not attribute the archived result to augmentation.

In [ ]:
class BreastCancerDataset(Dataset):

    def __init__(self, root_dir, transform=None):
        """
        Custom Dataset Class
        root_dir: Data root directory, containing two subfolders: 0_N and 1_IC
        transform: Data Augmentation/Preprocessing
        """
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        self.labels = []
        normal_dir = os.path.join(root_dir, '0_N')
        if os.path.exists(normal_dir):
            for img_name in os.listdir(normal_dir):
                if img_name.endswith(('.png', '.jpg', '.jpeg')):
                    self.samples.append(os.path.join(normal_dir, img_name))
                    self.labels.append(0)
        cancer_dir = os.path.join(root_dir, '1_IC')
        if os.path.exists(cancer_dir):
            for img_name in os.listdir(cancer_dir):
                if img_name.endswith(('.png', '.jpg', '.jpeg')):
                    self.samples.append(os.path.join(cancer_dir, img_name))
                    self.labels.append(1)
        print(f'Data set loaded: {len(self.samples)}pictures')
        print(f'Normal sample size: {self.labels.count(0)}, Number of cancer samples: {self.labels.count(1)}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return (image, label)


In [ ]:
train_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(p=0.5), transforms.RandomVerticalFlip(p=0.5), transforms.RandomRotation(15), transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02), transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
val_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])


## 3. Transfer-learning model

ResNet-18 is initialized from ImageNet weights. Its classifier is replaced with a two-layer head with dropout. Initially only the new head has trainable parameters; later stages unfreeze layer3 and layer4. BatchNorm running statistics are not frozen.

In [ ]:
def create_model(num_classes=2, pretrained=True, freeze_layers=True):
    """
    Create pre-trained models and implement transfer learning strategies
    
    Parameter Description:
    - num_classes: Number of Classification Categories (Binary Classification: Normal vs. Invasive Cancer)
    - pretrained: Should ImageNet pre-trained weights be used?
    - freeze_layers: Should we freeze certain layers for fine-tuning?
    """
    model = models.resnet18(pretrained=pretrained)
    num_features = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(num_features, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_classes))
    if freeze_layers:
        for name, param in model.named_parameters():
            if 'fc' not in name:
                param.requires_grad = False
            else:
                param.requires_grad = True
        print('Phase 1: Freeze all convolutional layers and train only the fully connected layers.')
    print('\n' + '=' * 50)
    print('Pre-trained CNN Architecture Information:')
    print('=' * 50)
    print(f'Model Name: ResNet-18')
    print(f'Parameter quantity: {sum((p.numel() for p in model.parameters())):,}')
    print(f'Trainable parameters: {sum((p.numel() for p in model.parameters() if p.requires_grad)):,}')
    print(f'Input size: 224×224×3')
    print(f'Output size: {num_classes} (Binary Classification)')
    print('=' * 50)
    return model


## 4. Training and reporting helpers

Cross-entropy loss, AdamW, cosine annealing and early stopping are retained from the original implementation. The unfreezing condition resets the optimizer whenever it is met. The final internal report uses last-epoch predictions, even after the best checkpoint is restored; see the method notes.

In [ ]:
def plot_training_history(history):
    """Plot Training History Curve"""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['train_loss'], label='Training loss')
    axes[0].plot(history['val_loss'], label='Verification loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training and validation loss')
    axes[0].legend()
    axes[0].grid(True)
    axes[1].plot(history['train_acc'], label='Training accuracy')
    axes[1].plot(history['val_acc'], label='Verification Accuracy Rate')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Training and validation accuracy')
    axes[1].legend()
    axes[1].grid(True)
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
def print_final_report(model, val_loader, device, all_preds, all_labels):
    """Print the Final Evaluation Report"""
    print('\n' + '=' * 50)
    print('Final Evaluation Report:')
    print('=' * 50)
    cm = confusion_matrix(all_labels, all_preds)
    print(f'Confusion matrix:\n{cm}')
    report = classification_report(all_labels, all_preds, target_names=['Normal', 'Invasive carcinoma'])
    print(f'\nClassification Report:\n{report}')
    tn, fp, fn, tp = cm.ravel()
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    precision = tp / (tp + fp)
    print(f'accuracy: {accuracy:.4f}')
    print(f'sensitivity: {sensitivity:.4f}')
    print(f'specificity: {specificity:.4f}')
    print(f'precision: {precision:.4f}')
    print('=' * 50)


In [ ]:
def train_model(model, train_loader, val_loader, num_epochs=25, device='cuda'):
    """
    Train the model
    
    Training Configuration Instructions:
    - Loss function: Cross-entropy loss 
    - Optimizer: AdamW
    - Learning Rate Scheduler: Cosine Annealing
    - Batch size: 16-32
    - Training Rounds: 25 rounds (adjustable based on early-stopping strategy)
    """
    if device == 'cuda' and torch.cuda.is_available():
        device = torch.device('cuda')
        print(f'using GPU: {torch.cuda.get_device_name(0)}')
    else:
        device = torch.device('cpu')
        print('using CPU')
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    params_to_update = []
    for name, param in model.named_parameters():
        if param.requires_grad:
            params_to_update.append(param)
            print(f'Training Layer: {name}')
    optimizer = optim.AdamW(params_to_update, lr=0.0001, weight_decay=0.0001)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-06)
    best_val_acc = 0.0
    patience = 5
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    print('\n' + '=' * 50)
    print('Training Configuration Information:')
    print('=' * 50)
    print(f'Loss function: CrossEntropyLoss')
    print(f'Optimizer: AdamW (lr=1e-4, weight_decay=1e-4)')
    print(f'Learning Rate Scheduling: CosineAnnealingLR')
    print(f'Number of training cycles: {num_epochs}')
    print(f'Early Stop Patience Value: {patience}')
    print('=' * 50 + '\n')
    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch + 1}/{num_epochs}')
        print('-' * 30)
        model.train()
        running_loss = 0.0
        running_corrects = 0
        total_samples = 0
        train_pbar = tqdm(train_loader, desc='Training')
        for inputs, labels in train_pbar:
            inputs = inputs.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            total_samples += inputs.size(0)
            train_pbar.set_postfix({'loss': loss.item()})
        epoch_loss = running_loss / total_samples
        epoch_acc = running_corrects.double() / total_samples
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc.item())
        model.eval()
        val_running_loss = 0.0
        val_running_corrects = 0
        val_total_samples = 0
        all_preds = []
        all_labels = []
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc='Verification')
            for inputs, labels in val_pbar:
                inputs = inputs.to(device)
                labels = labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
                val_running_loss += loss.item() * inputs.size(0)
                val_running_corrects += torch.sum(preds == labels.data)
                val_total_samples += inputs.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        val_loss = val_running_loss / val_total_samples
        val_acc = val_running_corrects.double() / val_total_samples
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc.item())
        print(f'Training Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
        print(f'Verification Loss: {val_loss:.4f} Acc: {val_acc:.4f}')
        scheduler.step()
        if val_acc > 0.8 and epoch > 5:
            print('Phase Two: Thaw the final two residual blocks for fine-tuning.')
            for name, param in model.named_parameters():
                if 'layer3' in name or 'layer4' in name:
                    param.requires_grad = True
            optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-05, weight_decay=0.0001)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs - epoch, eta_min=1e-06)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), CHECKPOINT_PATH)
            print(f'Model saved. Validation accuracy: {val_acc:.4f}')
            patience_counter = 0
        else:
            patience_counter += 1
            print(f'Early Stop Counter: {patience_counter}/{patience}')
        if patience_counter >= patience:
            print('Early stop triggered, training halted')
            break
    model.load_state_dict(torch.load(CHECKPOINT_PATH))
    plot_training_history(history)
    print_final_report(model, val_loader, device, all_preds, all_labels)
    return (model, history)


## 5. Training entry point

The 690-image training directory is divided into 552 training and 138 internal-validation images using the original seeded image-level split. The shared-transform behavior is preserved for traceability, not recommended as a pattern for a new experiment. This cell launches training and may download ImageNet weights.

In [ ]:
def main():
    torch.manual_seed(42)
    np.random.seed(42)
    train_dir = str(DATA_DIR / 'HW6training')
    train_dataset = BreastCancerDataset(train_dir, transform=train_transform)
    train_size = int(0.8 * len(train_dataset))
    val_size = len(train_dataset) - train_size
    train_subset, val_subset = torch.utils.data.random_split(train_dataset, [train_size, val_size])
    val_subset.dataset.transform = val_transform
    train_loader = DataLoader(train_subset, batch_size=16, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_subset, batch_size=16, shuffle=False, num_workers=0)
    print(f'Training set size: {len(train_subset)}')
    print(f'Validation set size: {len(val_subset)}')
    model = create_model(num_classes=2, pretrained=True, freeze_layers=True)
    model, history = train_model(model=model, train_loader=train_loader, val_loader=val_loader, num_epochs=25, device='cuda' if torch.cuda.is_available() else 'cpu')
    return model
trained_model = main()


### Archived training history

This figure was extracted from the submitted notebook. The recorded CPU run stopped after 18 epochs; it was not rerun for this repository.

![Original training history](../figures/training_history_original.png)

## 6. Course-provided validation and unlabeled prediction

The following block evaluates the separate 82-image validation folder using the model restored by training. It is separate from the 138-image internal split. An existing checkpoint is required if the training section is skipped.

In [ ]:
print('=== Begin validation set evaluation ===')
validation_dir = str(DATA_DIR / 'HW6validation')
val_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
if 'trained_model' not in locals():
    print('Reload the model...')
    model = models.resnet18(pretrained=False)
    num_features = model.fc.in_features
    model.fc = torch.nn.Sequential(torch.nn.Dropout(0.5), torch.nn.Linear(num_features, 256), torch.nn.ReLU(), torch.nn.Dropout(0.3), torch.nn.Linear(256, 2))
    model.load_state_dict(torch.load(CHECKPOINT_PATH))
    model.eval()
    if torch.cuda.is_available():
        model = model.cuda()
    trained_model = model
    print('Model loading complete！')
else:
    print('Use existing trained models')
    trained_model.eval()
print('\nBegin prediction on the validation set...')
predictions = []
true_labels = []
filenames = []
normal_dir = os.path.join(validation_dir, '0_N')
if os.path.exists(normal_dir):
    normal_files = [f for f in os.listdir(normal_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f'Find {len(normal_files)} normal pictures')
    for img_name in normal_files:
        img_path = os.path.join(normal_dir, img_name)
        img = Image.open(img_path).convert('RGB')
        img_tensor = val_transform(img).unsqueeze(0)
        with torch.no_grad():
            if torch.cuda.is_available():
                img_tensor = img_tensor.cuda()
            outputs = trained_model(img_tensor)
            _, pred = torch.max(outputs, 1)
        predictions.append(pred.item())
        true_labels.append(0)
        filenames.append(img_name)
cancer_dir = os.path.join(validation_dir, '1_IC')
if os.path.exists(cancer_dir):
    cancer_files = [f for f in os.listdir(cancer_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f'Find {len(cancer_files)} invasive carcinoma pictures')
    for img_name in cancer_files:
        img_path = os.path.join(cancer_dir, img_name)
        img = Image.open(img_path).convert('RGB')
        img_tensor = val_transform(img).unsqueeze(0)
        with torch.no_grad():
            if torch.cuda.is_available():
                img_tensor = img_tensor.cuda()
            outputs = trained_model(img_tensor)
            _, pred = torch.max(outputs, 1)
        predictions.append(pred.item())
        true_labels.append(1)
        filenames.append(img_name)
print(f'\ntotal {len(predictions)} pictures')
print('\n' + '=' * 50)
print('Validation Set Performance Report')
print('=' * 50)
accuracy = accuracy_score(true_labels, predictions)
print(f'Accuracy: {accuracy:.4f} ({int(accuracy * len(true_labels))}/{len(true_labels)} )')
cm = confusion_matrix(true_labels, predictions)
print(f'\nConfusion Matrix')
print(f'           Prediction normal  Prediction carcinoma')
print(f'Actually normal     {cm[0, 0]:4d}       {cm[0, 1]:4d}')
print(f'Actually carcinoma     {cm[1, 0]:4d}       {cm[1, 1]:4d}')
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn) if tp + fn > 0 else 0
specificity = tn / (tn + fp) if tn + fp > 0 else 0
precision = tp / (tp + fp) if tp + fp > 0 else 0
print(f'\nDetailed Indicators')
print(f'- Sensitivity (Recall): {sensitivity:.4f} (The ability to accurately identify cancer)')
print(f'- Specificity: {specificity:.4f} (The ability to correctly identify normal tissue)')
print(f'- Precision rate: {precision:.4f} (The proportion of samples predicted to have cancer that are actually cancerous)')
print(f'\nClassification Report:')
report = classification_report(true_labels, predictions, target_names=['Normal(N)', 'Invasive carcinoma(IC)'])
print(report)
plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = np.arange(2)
plt.xticks(tick_marks, ['N', 'IC'])
plt.yticks(tick_marks, ['N', 'IC'])
thresh = cm.max() / 2.0
for i in range(2):
    for j in range(2):
        plt.text(j, i, format(cm[i, j], 'd'), ha='center', va='center', color='white' if cm[i, j] > thresh else 'black')
plt.ylabel('Authenticity Label')
plt.xlabel('Prediction Label')
plt.tight_layout()
plt.show()


### Unlabeled test images

The 149 images are sorted by numeric filename identifier. The original saved prediction sequence is archived separately. This block writes a header-free prediction CSV for the course format. CPU inference matches the recorded run; image failures now stop execution instead of being assigned normal labels.

In [ ]:
if 'trained_model' in locals():
    trained_model = trained_model.cpu().eval()
print('=== Begin testing set prediction ===')
test_dir = str(DATA_DIR / 'HW6testing')
print(f'Test Set Path: {test_dir}')
if not os.path.exists(test_dir):
    print(f'Error: Test set path does not exist: {test_dir}')
    print('Please ensure the HW6testing folder exists.')
else:
    print(f'The test set folder exists. Processing has begun....')
    test_files = []
    for f in os.listdir(test_dir):
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff')):
            test_files.append(f)
    print(f'Found {len(test_files)} pictures')

    def extract_number(filename):
        try:
            name_without_ext = os.path.splitext(filename)[0]
            number_str = name_without_ext.split('_')[1]
            return int(number_str)
        except:
            return float('inf')
    test_files.sort(key=extract_number)
    print(f'Top 10 files: {test_files[:10]}')
    print(f'Last 10 files: {test_files[-10:]}')
    if 'trained_model' not in locals():
        print('Reload the model...')
        from torchvision import models
        import torch.nn as nn
        model = models.resnet18(pretrained=False)
        num_features = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(num_features, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 2))
        model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location='cpu'))
        model.eval()
        trained_model = model
        print('Model loaded!')
    test_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
    print('\nBegin predicting the test set...')
    predictions = []
    probabilities = []
    for i, filename in enumerate(test_files):
        img_path = os.path.join(test_dir, filename)
        try:
            img = Image.open(img_path).convert('RGB')
            img_tensor = test_transform(img).unsqueeze(0)
            with torch.no_grad():
                outputs = trained_model(img_tensor)
                probs = torch.nn.functional.softmax(outputs, dim=1)
                _, pred = torch.max(outputs, 1)
            predictions.append(pred.item())
            probabilities.append(probs.numpy()[0])
            if (i + 1) % 20 == 0:
                print(f'Processed {i + 1}/{len(test_files)} pictures')
        except Exception as e:
            raise RuntimeError(f'Failed to process {filename}') from e
    print(f'have processed {len(predictions)} pictures')
    print(f'\nPredictive Statistics:')
    print(f'- 0 (N): {predictions.count(0)} pictures')
    print(f'- 1 (IC): {predictions.count(1)} pictures')
    if len(predictions) != 149:
        print(f'Warning: Only {len(predictions)} image was predicted, but the assignment requires 149 images.')
        print(f'If the count is incorrect, please check the number of files in the test set folder.')
    print(f'\n' + '=' * 50)
    print('Prediction Result Sequence (0=Normal, 1=Invasive Cancer)')
    print('=' * 50)
    predictions_str = ','.join(map(str, predictions))
    print(predictions_str)
    pd.DataFrame(predictions).to_csv(RESULT_DIR / 'test_predictions.csv', index=False, header=False, encoding='utf-8-sig')
    print(f'\nPrediction results have been saved to: test_predictions.csv')


## 7. Grad-CAM examples

Grad-CAM targets `layer4[-1].conv2`. The first two images returned from each class directory are visualized; directory ordering is not fixed. These are examples from the training pool, not held-out localization evaluation. The displayed probabilities are softmax scores, not confidence intervals. The original backward-hook implementation is retained and may require adaptation for other PyTorch versions.

In [ ]:
if 'trained_model' in locals():
    trained_model = trained_model.cpu().eval()
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
import warnings
print('=== Grad-CAM  ===')
if 'trained_model' not in locals():
    print('Loading Model...')
    weights = models.ResNet18_Weights.IMAGENET1K_V1
    model = models.resnet18(weights=weights)
    num_features = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(num_features, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 2))
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location='cpu'))
    trained_model = model
    print('Model loaded!')
else:
    trained_model.eval()
trained_model.eval()


In [ ]:
class GradCAM:
    """Grad-CAM Visualization Class"""

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):

        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()
        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_backward_hook(backward_hook)

    def generate_cam(self, input_image, target_class=None):
        """
        Generate Grad-CAM heatmaps
        """
        output = self.model(input_image)
        if target_class is None:
            target_class = torch.argmax(output, dim=1).item()
        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, target_class] = 1
        output.backward(gradient=one_hot, retain_graph=True)
        gradients = self.gradients
        activations = self.activations
        weights = torch.mean(gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * activations, dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-08)
        return (cam, target_class, output)
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
target_layer = trained_model.layer4[-1].conv2
print(f'Target Layer: {target_layer}')
grad_cam = GradCAM(trained_model, target_layer)

def find_image_files(folder_path, max_images=2):
    """Search for image files in the folder"""
    if not os.path.exists(folder_path):
        print(f'Folder does not exist: {folder_path}')
        return []
    image_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif')
    images = []
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(image_extensions):
            full_path = os.path.join(folder_path, filename)
            images.append(full_path)
            if len(images) >= max_images:
                break
    return images
train_dir = str(DATA_DIR / 'HW6training')
image_paths = []
normal_folder = os.path.join(train_dir, '0_N')
normal_images = find_image_files(normal_folder, max_images=2)
if normal_images:
    print(f'Find {len(normal_images)} normal pictures')
    image_paths.extend(normal_images)
else:
    print("Can't find normal pictures")
cancer_folder = os.path.join(train_dir, '1_IC')
cancer_images = find_image_files(cancer_folder, max_images=2)
if cancer_images:
    print(f'Find {len(cancer_images)} invasive carcinoma pictures')
    image_paths.extend(cancer_images)
else:
    print("Can't find IC pictures")
print(f'\n {len(image_paths)} image was selected for visualization')
for i, path in enumerate(image_paths):
    folder_name = os.path.basename(os.path.dirname(path))
    label = 'Normal' if folder_name == '0_N' else 'Invasive Carcinoma'
    print(f'{i + 1}. {os.path.basename(path)} ({label})')


In [ ]:
def visualize_grad_cam(image_path, grad_cam, transform, model, figsize=(15, 5)):
    """Visualization of Grad-CAM results for a single image"""
    try:
        image = Image.open(image_path).convert('RGB')
        original_image = np.array(image)
        input_tensor = transform(image).unsqueeze(0)
        cam, target_class, output = grad_cam.generate_cam(input_tensor)
        probabilities = torch.nn.functional.softmax(output, dim=1)[0]
        predicted_class = torch.argmax(probabilities).item()
        predicted_prob = probabilities[predicted_class].item()
        cam = cam.squeeze().numpy()
        from PIL import Image as PILImage
        cam_resized = np.array(PILImage.fromarray(cam).resize((original_image.shape[1], original_image.shape[0]), PILImage.BILINEAR))
        heatmap = plt.cm.jet(cam_resized)[:, :, :3]
        heatmap = (heatmap * 255).astype(np.uint8)
        alpha = 0.5
        original_float = original_image.astype(float) / 255.0
        heatmap_float = heatmap.astype(float) / 255.0
        superimposed = original_float * (1 - alpha) + heatmap_float * alpha
        superimposed = np.clip(superimposed, 0, 1)
        superimposed = (superimposed * 255).astype(np.uint8)
        folder_name = os.path.basename(os.path.dirname(image_path))
        true_label = 0 if folder_name == '0_N' else 1
        label_names = {0: 'Normal', 1: 'Invasive Carcinoma'}
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        axes[0].imshow(original_image)
        axes[0].set_title(f'Original image\nTrue Label: {label_names[true_label]}')
        axes[0].axis('off')
        axes[1].imshow(cam_resized, cmap='hot')
        axes[1].set_title('CAM raw image')
        axes[1].axis('off')
        axes[2].imshow(heatmap)
        axes[2].set_title('Grad-CAM Heatmap')
        axes[2].axis('off')
        axes[3].imshow(superimposed)
        axes[3].set_title(f'Overlay Image\nPredicted: {label_names[predicted_class]} ({predicted_prob:.2%})')
        axes[3].axis('off')
        plt.tight_layout()
        plt.show()
        return {'true_label': true_label, 'predicted_class': predicted_class, 'predicted_prob': predicted_prob, 'cam': cam_resized, 'success': True}
    except Exception as e:
        print(f'error occur when {os.path.basename(image_path)} : {e}')
        return {'success': False, 'error': str(e)}


In [ ]:
if image_paths:
    print('\nGenerate Grad-CAM visualizations...')
    all_results = []
    for i, img_path in enumerate(image_paths):
        print(f'\nProcess images {i + 1}/{len(image_paths)}: {os.path.basename(img_path)}')
        result = visualize_grad_cam(img_path, grad_cam, transform, trained_model)
        if result['success']:
            all_results.append(result)
    if all_results:
        print('\n' + '=' * 60)
        print('Grad-CAM Analysis and Summary')
        print('=' * 60)
        for i, (img_path, result) in enumerate(zip(image_paths[:len(all_results)], all_results)):
            if not result['success']:
                continue
            label_names = {0: 'N', 1: 'IC'}
            true_label = result['true_label']
            pred_label = result['predicted_class']
            pred_prob = result['predicted_prob']
            print(f'\nLabel {i + 1}: {os.path.basename(img_path)}')
            print(f'  True Category: {label_names[true_label]}')
            print(f'  Prediction Category: {label_names[pred_label]} (Predicted-class probability: {pred_prob:.2%})')
            print(f"  Prediction correct: {('Y' if true_label == pred_label else 'N')}")
            cam = result['cam']
            mean_heat = np.mean(cam)
            max_heat = np.max(cam)
            print(f'  Heatmap Statistics:')
            print(f'    - Average Heat: {mean_heat:.4f}')
            print(f'    - Maximum Heat: {max_heat:.4f}')
    print('\nSave the comprehensive visualization results...')
    n_images = len([r for r in all_results if r['success']])
    if n_images > 0:
        fig, axes = plt.subplots(n_images, 4, figsize=(20, 5 * n_images))
        if n_images == 1:
            axes = axes.reshape(1, -1)
        row = 0
        for i, img_path in enumerate(image_paths):
            if i >= len(all_results) or not all_results[i]['success']:
                continue
            image = Image.open(img_path).convert('RGB')
            original_image = np.array(image)
            input_tensor = transform(image).unsqueeze(0)
            cam, target_class, output = grad_cam.generate_cam(input_tensor)
            cam_np = cam.squeeze().numpy()
            from PIL import Image as PILImage
            cam_resized = np.array(PILImage.fromarray(cam_np).resize((original_image.shape[1], original_image.shape[0]), PILImage.BILINEAR))
            heatmap = plt.cm.jet(cam_resized)[:, :, :3]
            heatmap = (heatmap * 255).astype(np.uint8)
            alpha = 0.5
            original_float = original_image.astype(float) / 255.0
            heatmap_float = heatmap.astype(float) / 255.0
            superimposed = original_float * (1 - alpha) + heatmap_float * alpha
            superimposed = np.clip(superimposed, 0, 1)
            superimposed = (superimposed * 255).astype(np.uint8)
            folder_name = os.path.basename(os.path.dirname(img_path))
            true_label = 0 if folder_name == '0_N' else 1
            label_names = {0: 'Normal', 1: 'Invasive Carcinoma'}
            probabilities = torch.nn.functional.softmax(output, dim=1)[0]
            predicted_class = torch.argmax(probabilities).item()
            axes[row, 0].imshow(original_image)
            axes[row, 0].set_title(f'Original image\nTrue: {label_names[true_label]}')
            axes[row, 0].axis('off')
            axes[row, 1].imshow(cam_resized, cmap='hot')
            axes[row, 1].set_title('CAM raw image')
            axes[row, 1].axis('off')
            axes[row, 2].imshow(heatmap)
            axes[row, 2].set_title('Grad-CAM Heatmap')
            axes[row, 2].axis('off')
            axes[row, 3].imshow(superimposed)
            axes[row, 3].set_title(f'Overlay Result\nPrediction: {label_names[predicted_class]}')
            axes[row, 3].axis('off')
            row += 1
        plt.tight_layout()
        plt.savefig(RESULT_DIR / 'grad_cam_results.png', dpi=300, bbox_inches='tight')
        print(f'Visualization results have been saved to: grad_cam_results.png')
        print('\n' + '=' * 60)
        print('Grad-CAM Visualization Summary')
        print('=' * 60)
        print('The red area indicates the region of primary focus for the model.')
else:
    print('No image files were found for visualization.')
print('\n=== Grad-CAM visualization completed ===')


### Archived Grad-CAM panels

Original saved panels, not newly generated results. Heatmaps illustrate regions contributing to the selected class score; they are not validated tumour masks. Image provenance and reuse permissions remain unspecified in the course materials.

![Original Grad-CAM examples](../figures/grad_cam_examples_original.png)

## Limitations and next steps

A new experiment should separate training/validation dataset transforms, use a one-time unfreezing schedule, recompute checkpoint metrics, and split by patient or slide where identifiers are available. Those changes would require retraining and fresh results. They have not been implemented as if already validated here.

See [method notes](../docs/method_notes.md) for the full distinction between original behavior and editorial changes.